#### Cellene under er hentet fra: https://github.com/HVL-ML/DAT255/blob/main/notebooks/15_gradio_and_streamlit.ipynb

# Gradio (and Streamlit) deployment

For deploying an ML-based app there are various approaches, but [Gradio](gradio.app) and [Streamlit](https://streamlit.io/) are both quick and convenient ways to do so. Typically we would implement this in a plain python (`.py`) file rather than a `.ipynb` notebook, but Gradio works here too, so let's try that first. Streamlit needs to be run directly in python, but the code is given below, so you can try out that too.

In this example we set up an image classifier, where the user can upload an image and get the top 5 class predictions in return.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions,
)
from PIL import Image

2026-03-10 13:38:23.380367: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-10 13:38:23.469848: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-10 13:38:23.469923: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-10 13:38:23.538299: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-10 13:38:25.715985: W tensorflow/compiler/tf

## Set up a pretrained model

Let's download and set up a `MobileNetV2` model, trained on the 1000 classes in the ImageNet dataset. You can change this to anything you like. Remeber, however, to also use the appropriate preprocessing function.

In [89]:
# model = MobileNetV2(weights="imagenet")

# The model loading below is from Deep Learning with Python, Third edition. Chapter: 8, Fitting the model
model = keras.models.load_model(
    "../models/hierarchical_cnn_baseline_hybrid_crop.keras"
)

def classify_image(img: Image.Image):

    # Resize to the input image to what MobileNet expects
    img = img.resize((300, 300))
    arr = np.array(img)

    # Check the color channels, so we can take both grayscale, RGB, and RGBA images as input.
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 4:
        arr = arr[..., :3]

    # Add batch dim, and preprocess
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr.astype("float32"))

    # Predict!
    preds = model.predict(arr, verbose=0)
    # print(preds)
    # print(preds["lvl1"])
    # top5 = decode_predictions(preds["lvl1"], top=1)[0]  # [(id, label, prob), ...]
    # print(top5)
    return preds

    # return {label: float(prob) for (_, label, prob) in top5}



In [94]:
from matplotlib import pyplot as plt

def classify_image_twice(img: Image.Image):
    plt.imshow(img)
    predictions = classify_image(img)
    print(predictions)
    return predictions

## Set up the Gradio interface

Check the [documentation](https://www.gradio.app/docs) for the various things we can add here.

In [95]:
import gradio as gr

# Example images
examples = [
    "https://cdn.britannica.com/79/232779-050-6B0411D7/German-Shepherd-dog-Alsatian.jpg",
    "https://cdn.britannica.com/41/126641-050-E1CA0E61/cat-suns-hill-Parthenon-Athens-Greece-Acropolis.jpg",
]

with gr.Blocks(title="Visual vehicle recognition in varying lighting conditions") as demo:
    gr.Markdown("## Visual vehicle recognition in varying lighting conditions")
    gr.Markdown(
        "Upload an image or click an example below. "
    )

    with gr.Row():
        image_input = gr.Image(type="pil", label="Input Image")
        label_output = [{},{},{},{}]
        label_output[0] = gr.Label(num_top_classes=1, label="Bilmerke")
        label_output[1] = gr.Label(num_top_classes=3, label="Bilmodell")
        label_output[2] = gr.Label(num_top_classes=3, label="Årsperiodemodell")
        label_output[3] = gr.Label(num_top_classes=3, label="Farge")

    classify_btn = gr.Button("Classify", variant="primary")
    classify_btn.click(fn=classify_image_twice, inputs=image_input, outputs=label_output)

    gr.Examples(
        examples=examples,
        inputs=image_input,
        outputs=label_output,
        fn=classify_image_twice,
        cache_examples=True
    )

Now we can run it:

In [97]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/gradio/blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/gradio/blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/anyio/to_thread.py", line 63, in

Caching examples at: '/home/jkaste03/school/dat191/visual-vehicle-recognition-varying-lighting-conditions/gradio/.gradio/cached_examples/270'
{'lvl1': array([[4.7506057e-07]], dtype=float32), 'lvl2': array([[5.9806862e-09, 9.9166267e-14, 3.6385860e-08, 2.3825676e-16,
        6.1428955e-15, 1.0000000e+00, 1.9387057e-17]], dtype=float32)}


Traceback (most recent call last):
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/gradio/blocks.py", line 2168, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jkaste03/venvs/tf-2.16/lib/python3.12/site-packages/gradio/blocks.py", line 1876, in postprocess_data
    self.validate_outputs(block_fn, predictions)  # type: ignore
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jkaste03/venvs/tf-2